# **Project Name**    -  PRCP-1022-HospitalStayPred

##### **Project Type**    - Classification
##### **Contribution**    - Individual

# **Project Summary -**

**Dataset**: 318,438 patient admission records from a network of hospitals across multiple regions  
**Time Period**: Aggregated historical admissions (no temporal sequencing)  
**Key Metrics**:
- **Target Variable**: Length of Stay (`Stay`) — ordinal categorical variable with 11 classes:
  - **Short-term** (0-20 days): ~45% of admissions
  - **Medium-term** (21-50 days): ~40% of admissions  
  - **Long-term** (51-100 days): ~12% of admissions
  - **Extended care** (>100 days): ~3% of admissions
  
- **Class Imbalance Ratio**: 31.9:1 (largest class 21-30 days: 87,491 patients vs. smallest class >100 days: ~2,740 patients)

**Features** (18 total):
- Hospital infrastructure: Hospital code, type, region, available extra rooms, ward type/facility, bed grade
- Patient demographics: Age (10-year brackets), city code
- Admission context: Type of admission (Emergency/Trauma/Urgent), severity of illness (Minor/Moderate/Extreme), department
- Financial: Admission deposit (₹1,800–11,000)
- Social: Number of visitors (0-32)

# **GitHub Link -**

https://github.com/priyankamanik35-lgtm/Hospital_Stay_Duration

# **Problem Statement**


Hospitals operate as complex organizations balancing **resource availability, patient care quality, and operational efficiency**. One of the most critical challenges in hospital management is predicting how long individual patients will require hospitalization. Accurate length-of-stay predictions enable hospitals to:

- **Optimize bed allocation and ward capacity planning** — Knowing which patients will require 0-10 days vs. >100 days helps prioritize expensive ICU/critical-care beds for acute cases and reserve general ward beds for routine recoveries.
- **Forecast staffing requirements** — Long-stay cases demand specialized care protocols and higher nursing ratios; predicting LOS allows hospitals to schedule specialized personnel proactively.
- **Improve financial planning** — Advance deposit policies can be dynamically set based on predicted stay duration; overbooking risk for surgical suites and diagnostic facilities can be better managed.
- **Reduce readmission and mortality risk** — Early identification of patients likely to have extended stays enables preventive interventions, earlier discharge planning, and specialized monitoring.



# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, ttest_ind

from sklearn.preprocessing import LabelEncoder, StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV,StratifiedKFold

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import pickle

import warnings
warnings.filterwarnings("ignore")

### Dataset Loading

In [ ]:
# Load Dataset
df = pd.read_csv('HealthCare_Dataset.csv')

### Dataset First View

In [ ]:
# Dataset First Look
df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
df.shape

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
df.duplicated().sum()

#### Missing Values/Null Values

In [ ]:
# Missing / Null Values count per column
null_summary = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df))  100
})
null_summary[null_summary['Missing_Count'] > 0]


### What did you know about your dataset?

* **Volume**: The dataset contains **318,438 patient admission records** across **18 columns**.
* **Identifiers**: `case_id` represents unique admission transactions, while `patientid` identifies unique individuals (allowing recurrent patient tracking).
* **Missing Data**: Missing values exist exclusively in `Bed_Grade` (113 rows, ~0.04%) and `City_Code_Patient` (4,532 rows, ~1.42%). All other operational and demographic fields are completely populated.
* **Target Variable**: `Stay` represents categorical length of stay intervals (e.g., 0-10, 11-20, 21-30, up to More than 100 Days). Note: The raw interval 'Nov-20' is an Excel auto-formatting typo for '11-20' and will be standardized during wrangling.
* **Data Types**: 7 integer features, 2 float features, and 9 categorical string features.


## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns

In [ ]:
# Dataset Describe
df.describe()

In [ ]:
# Also checking summary stats for categorical columns - shows unique count, top category, and its frequency
df.describe(include='object')

### Variables Description

* **case_id**: Unique identifier for each hospital admission case.
* **Hospital_code**: Integer identifier for the specific hospital branch.(1 to 32)
* **Hospital_type_code**: Categorical code designating the classification of the hospital (a, b, c, d, e, f, g).
* **City_Code_Hospital**: Geographic municipality code where the hospital is situated.(1 to 13)
* **Hospital_region_code**: Regional zone identifier of the hospital (X, Y, Z).
* **Available_Extra_Rooms_in_Hospital**: Number of unoccupied extra rooms available at the time of patient intake.(0 to 24)
* **Department**: Medical specialization overseeing care (gynecology, anesthesia, radiotherapy, TB & Chest disease, surgery).
* **Ward_Type**: Hospital ward classification (P, Q, R, S, T, U).
* **Ward_Facility_Code**: Operational identifier of the ward infrastructure (A, B, C, D, E, F).
* **Bed_Grade**: Structural quality/tier of the assigned bed (1 to 4).
* **patientid**: Unique patient identification number.
* **City_Code_Patient**: Resident municipality code of the patient.
* **Type_of_Admission**: Nature of admission triage (Emergency, Trauma, Urgent).
* **Severity_of_Illness**: Clinical severity assessment upon entry (Minor, Moderate, Extreme).
* **Visitors_with_Patient**: Count of registered visitors accompanying the patient.(0 to 32)
* **Age**: Age demographic bracket of the patient (e.g., 0-10, 11-20, ..., 91-100).
* **Admission_Deposit**: Advance deposit payment amount collected during admission (INR).
* **Stay**: **Target Variable** indicating the length of hospital stay in days.(0-10 to more than 100 days)


### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
df.nunique()

In [ ]:
# Checking exact category labels for Stay and Age - important since these are ordinal
# and we need to preserve their natural order during encoding
print("Stay categories:", sorted(df['Stay'].unique()))

In [ ]:
print("Age categories :", sorted(df['Age'].unique()))

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
df_clean = df.copy()
df_clean['Patient_Admission_Count'] = df.groupby('patientid')['case_id'].transform('count')

In [ ]:
# Dropping unnecessary columns
df_clean.drop(['case_id', 'patientid'],axis=1,inplace=True)

In [ ]:
# correcting wrong inputs
df_clean['Age'] = df_clean['Age'].replace({'Nov-20':'11-20'})
df_clean['Stay'] = df_clean['Stay'].replace({'Nov-20':'11-20'})

In [ ]:
# Filling null values
df_clean['Bed_Grade'] .fillna(df_clean['Bed_Grade'].mode()[0],inplace=True)
df_clean['City_Code_Patient'].fillna(df_clean['City_Code_Patient'].mode()[0],inplace=True)

In [ ]:
# Convert Bed_Grade and City_Code_Patient to integers
df_clean['Bed_Grade'] = df_clean['Bed_Grade'].astype(int)
df_clean['City_Code_Patient'] = df_clean['City_Code_Patient'].astype(int)

### What all manipulations have you done and insights you found?

* **Dropping column**: Dropping case_id and patientid as it is not needed for predictive analysis.
* **Corrected Label Artifact**: Fixed `'Nov-20'` label into its true ordinal interval `'11-20'`, ensuring clean multi-class classification targets.
* **Mode Imputation**: Missing records in `Bed_Grade` (113 instances) and `City_Code_Patient` (4,532 instances) were cleanly imputed with respective modal values, avoiding row drop and data leakage.
* **Changing Dtype**: As we have only 2 columns with float value so for easy intrpretation we convert them to integer.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1 (Target Variable 'Stay' Distribution)

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(x='Stay', data=df_clean, palette='viridis')
plt.title('Distribution of Hospital Length of Stay (Target Variable)', fontweight='bold')
plt.xlabel('Length of Stay (Days)')
plt.ylabel('Number of Patients')
for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A countplot is chosen to visualize the class frequency distribution across all 11 ordinal target classes of Length of Stay.

##### 2. What is/are the insight(s) found from the chart?
The stay distribution is right-skewed. The majority of admissions fall into the **21-30 days** (87,491 patients) and **11-20 days** (78,139 patients) buckets, accounting for over 52% of all admissions. Stays exceeding 60 days represent long-tail clinical cases.

##### 3. Will the gained insights help creating a positive business impact?

Yes — this has direct positive business impact for hospital resource planning. Since ~70% of patients are discharged within 40 days, hospitals can use this baseline to plan bed turnover, staffing schedules, and short-stay ward capacity more efficiently. On the risk side, the small but persistent "More than 100 Days" group represents a negative-growth signal: these long-stay patients consume disproportionate hospital resources (beds, staff time, equipment) relative to their numbers, which can strain capacity and increase costs if not identified and managed proactively (e.g., through early intervention or specialized long-term care pathways). Also, this class imbalance means our ML models will likely need special handling later (e.g., appropriate evaluation metrics) so that rare-but-important long-stay cases aren't ignored.

#### Chart - 2 (Admissions by Medical Department)

In [ ]:
# Chart - 2 visualization code: Department Admissions
plt.figure(figsize=(8, 5))
sns.countplot(x='Department', data=df_clean, palette='crest')
plt.title('Patient Intake Volume Across Medical Departments', fontweight='bold')
plt.xlabel('Department')
plt.ylabel('Total Admissions')
for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?
A vertical bar chart accurately illustrates the disparity in workload and intake across specialized departments.

##### 2. What is/are the insight(s) found from the chart?
Gynecology dominates admissions (~249,000 cases), followed by Anesthesia and Radiotherapy. Surgery and TB/Chest diseases see comparatively fewer admissions.

##### 3. Will the gained insights help creating a positive business impact?
Hospitals can allocate capital, obstetrics equipment, and specialized personnel primarily to Gynecology while maintaining lean surgical scheduling.

#### Chart - 3 (Severity of Illness Breakdown)

In [ ]:
# Chart - 2 visualization code: Severity of Illness Pie Chart
plt.figure(figsize=(8, 8))
df_clean['Severity_of_Illness'].value_counts().plot.pie(
    autopct='%1.1f%%', colors=sns.color_palette('crest'), startangle=90)
plt.title('Proportion of Severity of Illness')
plt.ylabel('')
plt.show()


##### 1. Why did you pick the specific chart?
A pie chart with percentage callouts effectively shows the proportional composition of patient severity tiers at triage.

##### 2. What is/are the insight(s) found from the chart?
'Moderate' severity accounts for the largest proportion (~55.2%), followed by 'Minor' (~27.2%) and 'Extreme' (~17.5%).

##### 3. Will the gained insights help creating a positive business impact?
Yes — positive impact for resource allocation: knowing that ~55% of cases are Moderate severity helps hospitals right-size general ward capacity versus ICU/critical-care capacity, since Extreme cases (17.8%) likely require disproportionately more intensive resources (staff, equipment, monitoring) relative to their share of admissions. On the risk side, if Extreme-severity cases are also correlated with longer stays (which we'll verify in the bivariate section), under-provisioning critical-care capacity based on this ~18% share alone could be risky — severity doesn't scale linearly with resource need, so hospitals shouldn't naively allocate resources proportional to case-count percentages alone.

#### Chart - 4 (Age Group Demographics)

In [ ]:
# Chart - 4 visualization code: Age Distribution
plt.figure(figsize=(11, 5))
age_order = ['0-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81-90', '91-100']
sns.countplot(x='Age', data=df_clean, order=age_order, palette='viridis')
plt.title('Patient Distribution Across Age Brackets', fontsize=14, fontweight='bold')
plt.xlabel('Age Group', fontsize=12)
plt.ylabel('Patient Count', fontsize=12)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A count plot suits Age well since it's a bucketed/ordinal categorical variable, and we want to compare patient volume across age groups in order, from youngest to oldest.
##### 2. What is/are the insight(s) found from the chart?
The patient base is concentrated in the middle-aged to older-adult range — 31-40 and 41-50 are the two largest groups (~63,500 patients each), followed by 51-60 (~48,500). Volume tapers off on both ends: very young patients (0-10, 11-20) and the elderly (81-90, 91-100) make up a much smaller share, with 91-100 being the rarest (~1,000 patients). Interestingly, 71-80 has slightly more patients than 61-70, a small but notable break from an otherwise smooth decline.

##### 3. Will the gained insights help creating a positive business impact?
Yes, positive impact — hospitals can use this age concentration (30–60 age band) to tailor department staffing, equipment allocation, and even preventive health programs toward the working-age population that dominates admissions here. On the risk side, the very small sample sizes in 0-10 and 91-100 age brackets mean any model we train will have limited data to learn patterns for these groups — predictions for very young or very elderly patients may be less reliable, which is a real limitation to flag for a healthcare use case where under-serving edge-case age groups could have real consequences (e.g., an inaccurate stay-length prediction for an elderly patient could affect discharge planning or care allocation).

#### Chart - 5 (Admission Deposit Distribution)

In [ ]:
# Chart - 5 visualization code: Admission Deposit
plt.figure(figsize=(12, 6))
sns.histplot(data=df_clean, x='Admission_Deposit', bins=50, kde=True, color='teal')
plt.title('Distribution of Admission Deposit Amount')
plt.xlabel('Admission Deposit')
plt.ylabel('Frequency')
plt.show()

##### 1. Why did you pick the specific chart?
Admission_Deposit is a genuinely continuous numeric variable, so a histogram with a KDE overlay is ideal — it shows both the frequency distribution across bins and the smooth underlying shape, letting us spot skewness and outliers at a glance.
##### 2. What is/are the insight(s) found from the chart?
Admission deposits follow an approximately bell-shaped distribution centered around a median , spanning from ₹1,800 to ₹11,000+.

##### 3. Will the gained insights help creating a positive business impact?
Yes — positive impact for financial planning: hospitals can use this typical deposit range (₹3,000–6,500) as a benchmark for setting admission deposit policies and estimating expected cash flow from new admissions. The long right tail is worth investigating further, though — high-deposit admissions could correlate with higher severity cases or specific departments (something we'll check in the bivariate section). If unexplained, unusually high deposits with no clear justification could indicate inconsistent billing practices across hospitals, which would be a negative-growth/trust risk worth flagging to hospital administration for standardization.

#### Chart - 6 (Length of Stay across Severity Levels)

In [ ]:
# Chart - 6 visualization code: Stay vs Severity of Illness
plt.figure(figsize=(14, 6))
stay_order = ['0-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81-90', '91-100', 'More than 100 Days']
sns.countplot(x='Stay', hue='Severity_of_Illness', data=df_clean, order=stay_order, palette='Set2')
plt.title('Length of Stay by Severity of Illness')
plt.xlabel('Stay Duration')
plt.ylabel('Patient Count')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A grouped count plot lets us compare the volume of Moderate, Minor, and Extreme cases within each Stay duration bucket side by side, making it easy to see whether severity level shifts the shape of the stay-duration distribution.

##### 2. What is/are the insight(s) found from the chart?

Interestingly, Moderate severity is the tallest bar across almost every single stay bucket, simply because it's the largest overall category (55% of all patients, as seen in Chart 3 pie chart) — this proportional dominance carries through regardless of stay length. All three severity levels peak at the same bucket (21-30 days) and follow a similar right-skewed shape, which suggests that raw stay duration alone doesn't show an obvious separation by severity in count terms — this is a case where visualizing proportions (percentage within each severity group) would tell a clearer story than raw counts, something worth revisiting with a normalized version of this chart or in the hypothesis testing section.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This is actually a caution flag more than a clean win: if hospital planners assumed "Extreme severity = automatically longer stays," this raw chart alone doesn't strongly confirm that visually, since Moderate dominates every bucket by volume. Relying on this chart without normalizing by group size could lead to misleading conclusions and poor resource allocation decisions. The right move is to statistically test whether severity truly influences stay duration (which we'll do formally in the Hypothesis Testing section) rather than relying on visual count comparisons alone — this is a good example of why business decisions shouldn't be based on a single unnormalized chart.

#### Chart - 7 (Stay Duration by Admission Type)

In [ ]:
# Chart - 7 visualization code: Stay vs Type of Admission
plt.figure(figsize=(14, 6))
sns.countplot(x='Stay', hue='Type_of_Admission', data=df_clean, order=stay_order, palette='Accent')
plt.title('Length of Stay by Type of Admission (Emergency vs Trauma vs Urgent)', fontsize=14, fontweight='bold')
plt.xlabel('Length of Stay')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?
A grouped bar chart highlights the operational difference between Urgent, Trauma, Emergency admissions.

##### 2. What is/are the insight(s) found from the chart?
Trauma patients consistently demonstrate higher tendencies for 11-60+ day stays compared to Emergency or Urgent admissions.

##### 3. Will the gained insights help creating a positive business impact?
This is an important finding for both business and modeling purposes. On the business side, it confirms Trauma patients admission are resultings more staying days.Trauma centers can plan ward capacity dynamically based on seasonal or daily trauma admission rates.

#### Chart - 8 ( Average Visitors across Stay Categories)

In [ ]:
# Chart - 8 visualization code: Visitors vs Stay
plt.figure(figsize=(12, 5))
visitor_stay = df_clean.groupby('Stay')['Visitors_with_Patient'].mean().reindex(stay_order).reset_index()
sns.barplot(x='Stay', y='Visitors_with_Patient', data=visitor_stay, palette='flare')
plt.title('Average Number of Visitors by Length of Stay', fontweight='bold')
plt.xlabel('Length of Stay')
plt.ylabel('Average Visitors per Patient')
for p in plt.gca().patches:
    plt.gca().annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=9, xytext=(0, 2), textcoords='offset points')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?
A bar plot displaying group means illustrates the continuous trend between visitor frequency and duration of care.

##### 2. What is/are the insight(s) found from the chart?
Average visitors climb steadily from **2+ visitors** in the 0-10 day bracket to **7+ visitors** for stays >100 days.

##### 3. Will the gained insights help creating a positive business impact?
Strong visitor surges correlate with long stays. Regulating visitor hours and managing lobby traffic reduces nosocomial infection risks and improves ward sanitation.

#### Chart - 9 (Admission Deposit by Age Group)

In [ ]:
# Chart - 9 visualization code: Boxplot Admission Deposit by Age
plt.figure(figsize=(12, 5))
sns.boxplot(x='Stay', y='Admission_Deposit', data=df_clean, order=age_order, palette='coolwarm')
plt.title('Admission Deposit Distribution Across Patient Age Groups', fontweight='bold')
plt.xlabel('Stay Duration (days)')
plt.ylabel('Admission Deposit (INR)')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?
A box plot is the standard tool for comparing a continuous numeric variable (Admission_Deposit) across multiple categories (Stay buckets) — it shows median, interquartile spread, and outliers in one compact view, which is far more informative than just comparing means.

##### 2. What is/are the insight(s) found from the chart?
Surprisingly, the median admission deposit is remarkably stable across all stay-duration buckets, hovering consistently between ₹4,300 and ₹4,900 regardless of whether the stay is 0-10 days or More than 100 Days. The interquartile ranges also look similar in width across buckets. Every single bucket shows a large number of high-value outliers stretching up to ₹10,000+, and this outlier pattern doesn't intensify or shrink meaningfully as stay length increases. This is a fairly flat relationship — Admission_Deposit does not appear to scale with length of stay, which is somewhat counter-intuitive (one might expect deposits to be set higher in anticipation of longer stays).

##### 3. Will the gained insights help creating a positive business impact?
This is a useful negative finding: it suggests Admission_Deposit is likely set at time of admission based on factors other than expected stay duration (e.g., department, hospital policy, or a flat/near-flat institutional rate) — meaning hospitals may be under-collecting deposits for patients who end up staying much longer, and over-collecting relative to need for very short stays. From a business standpoint, this is actionable: hospitals could explore a more dynamic deposit-setting policy tied to predicted stay length (which is exactly what this ML project could support) to improve cash flow matching. From a modeling standpoint, it also tells us Admission_Deposit alone may be a weak predictor of Stay — we'll need to rely more on categorical features like Severity, Department, and Admission Type for prediction power, which we should verify through correlation/feature importance later.

#### Chart - 10 (Extra Room Availability by Hospital Region)

In [ ]:
# Chart - 10 visualization code: Extra Rooms by Region
plt.figure(figsize=(10,5))
sns.boxplot(x='Hospital_region_code', y='Available_Extra_Rooms_in_Hospital',hue='Severity_of_Illness', data=df_clean, palette='pastel')
plt.title('Available Extra Rooms by Hospital Regional Zone', fontweight='bold')
plt.xlabel('Hospital Region Code')
plt.ylabel('Available Extra Rooms')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?
A box plot with hue lets us compare the distribution of Available_Extra_Rooms_in_Hospital across both Hospital_region_code and Severity_of_Illness at once — useful for checking whether hospital capacity is allocated differently based on region or case severity.

##### 2. What is/are the insight(s) found from the chart?
The core distributions are nearly identical across all three regions (X, Y, Z) and all three severity levels — median available extra rooms sits consistently around 3, with the interquartile range roughly 2–4 in every single group. This tells us hospital room availability doesn't systematically vary by region or by the severity of the patient being admitted, which is somewhat expected since room availability is a hospital-level operational metric rather than something that would logically respond to individual patient severity. The one notable difference is in the outliers: Region Y shows several extreme outlier values (up to 24 extra rooms), which don't appear in Regions X or Z, suggesting Region Y may include a few larger or less-occupied hospitals in the dataset.

##### 3. Will the gained insights help creating a positive business impact?
This is primarily a "ruling out" insight rather than a strong positive finding — it tells us Available_Extra_Rooms_in_Hospital likely won't be a strong standalone predictor of Stay, since it doesn't vary meaningfully with severity (which we've already shown is stay-related). That's still valuable for modeling — it helps set expectations and guides feature selection later. On the operational side, the outlier hospitals in Region Y with unusually high room availability could be worth investigating separately — if genuine, they may represent underutilized capacity that could be redirected to relieve pressure on more constrained hospitals in other regions; if data-entry anomalies, they'd need cleaning before being trusted in any capacity-planning decision.

#### Chart 11 – (Ward Type Utilization vs Length of Stay)

In [ ]:
# Chart - 11 visualization code: Ward Type vs Stay
plt.figure(figsize=(14, 6))
sns.countplot(x='Stay', hue='Ward_Type', data=df_clean, order=stay_order, palette='Accent')
plt.title('Ward Type Allocation Across Length of Stay', fontweight='bold')
plt.xlabel('Length of Stay')
plt.ylabel('Number of Patients')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?
Grouped bar chart detailing how ward categories (P, Q, R, S, T, U) are utilized across short vs prolonged stays.

##### 2. What is/are the insight(s) found from the chart?
Wards **R** and **Q** accommodate the overwhelming majority of medium stays (11-30 days), whereas ward **S** and **P** handle extended acute cases.

##### 3. Will the gained insights help creating a positive business impact?
Supports ward conversion strategies: high-turnover general beds can be optimized in R & Q while specialized nursing is reserved in S & P.

#### Chart - 12 (Bed Grade Impact on Stay Duration)

In [ ]:
# Chart - 12 visualization code: Bed Grade Distribution across Stay
plt.figure(figsize=(14, 6))
sns.countplot(x='Stay', hue='Bed_Grade', data=df_clean, order=stay_order, palette='Set2')
plt.title('Bed Grade Allocation Across Length of Stay Categories', fontsize=14, fontweight='bold')
plt.xlabel('Length of Stay')
plt.ylabel('Count of Patients')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?
Segmented countplot to analyze the relationship between room luxury/tier (Bed Grade 1 to 4) and patient hospitalization length.

##### 2. What is/are the insight(s) found from the chart?
Bed Grade 2 and 3 account for the vast majority of all admissions across all stay lengths, while Grade 1 (basic) is more prominent in shorter stays (0-20 days).
Bed_Grade itself shows very little consistent effect on Stay_ordinal — medians stay roughly in the 2–3 range across all four bed grades (1.0 through 4.0), with no clear upward or downward trend as bed grade improves.
 What stands out more is stay duration. All the above chart mentioning less number of patients staying 60~100days for longer treatment or for sever cases.
##### 3. Will the gained insights help creating a positive business impact?
Guides facility upgrade investments toward Grade 2 and 3 beds which generate the highest volume and occupancy rates.
Mixed but useful insight — the weak relationship between Bed_Grade and Stay suggests bed quality alone isn't a strong lever for predicting or managing stay duration, which is somewhat reassuring (it implies stay length is driven more by patient condition than by room amenities).  it could point to differences in hospital efficiency, protocols, or patient case-mix that administrators should investigate further. This is a case where the insight is a starting point for a deeper investigation rather than an immediately actionable fix.

#### Chart - 13 (Admission Deposit by Admission Type & Severity)

In [ ]:
# Chart - 13 visualization code: Multivariate Boxplot
plt.figure(figsize=(10,4))
sns.barplot(x='Type_of_Admission', y='Admission_Deposit', hue='Severity_of_Illness', data=df_clean, palette='mako')
plt.title('Average Admission Deposit by Admission Type and Severity Level', fontweight='bold')
plt.xlabel('Type of Admission')
plt.ylabel('Average Admission Deposit (INR)')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?
A multivariate grouped bar chart demonstrates interactions between clinical urgency, severity level, and financial collections.

##### 2. What is/are the insight(s) found from the chart?
Urgent and Emergency admissions collect higher baseline deposits across all severity categories compared to Trauma admissions.

##### 3. Will the gained insights help creating a positive business impact?
Provides operational justification for standardized financial deposit tiers across admission routes without risking bad-debt write-offs.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
plt.figure(figsize=(10, 7))
num_cols = ['Hospital_code', 'City_Code_Hospital', 'Available_Extra_Rooms_in_Hospital', 'Bed_Grade',
            'City_Code_Patient', 'Visitors_with_Patient', 'Admission_Deposit']
corr_matrix = df_clean[num_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap of Key Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A correlation heatmap is the standard tool for quickly scanning linear relationships between all numeric/ordinal variable pairs at once, letting us spot which features are strongly related to our target (Stay_ordinal) and which are redundant with each other, without having to build individual scatter plots for every combination.

##### 2. What is/are the insight(s) found from the chart?

The numeric features show very weak correlations overall, meaning they don’t strongly influence each other.

Visitors_with_Patient vs Admission_Deposit → slight positive link (more visitors, slightly higher deposits).

Hospital_code vs City_Code_Hospital → mild correlation, expected since hospitals are tied to city codes.

Bed_Grade vs Admission_Deposit → almost no relationship, deposits don’t scale with bed quality.

Available_Extra_Rooms → independent, no meaningful correlation with other variables.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code (Sampled for execution performance)
sample_df = df_clean[['Available_Extra_Rooms_in_Hospital', 'Visitors_with_Patient', 'Admission_Deposit', 'Severity_of_Illness']].sample(1000, random_state=42)

sns.pairplot(df_clean[['Available_Extra_Rooms_in_Hospital', 'Visitors_with_Patient', 'Admission_Deposit', 'Severity_of_Illness']], hue='Severity_of_Illness', palette='Dark2', corner=True)
plt.suptitle('Pairwise Feature Distributions by Severity of Illness', y=1.02, fontsize=14, fontweight='bold')
plt.show()


##### 1. Why did you pick the specific chart?

I picked the pairplot because it’s a powerful way to visually explore relationships between multiple variables at once. Unlike a single correlation chart (which only shows numeric strength), the pairplot lets you actually see the scatter patterns, clusters, and distributions.

## ***5. Hypothesis Testing***

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis (H0):** There is no significant association between Severity of Illness and Length of Stay — the two variables are independent.

**Alternate Hypothesis (H1):** There is a significant association between Severity of Illness and Length of Stay — the two variables are not independent.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
# Using Chi-Square Test of Independence since both variables are categorical
from scipy.stats import chi2_contingency

contingency_table_1 = pd.crosstab(df_clean['Severity_of_Illness'], df_clean['Stay'])
chi2_stat, p_value_1, dof, expected = chi2_contingency(contingency_table_1)

print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"P-Value: {p_value_1:.10f}")
print(f"Degrees of Freedom: {dof}")

if p_value_1 < 0.05:
    print("\nReject the Null Hypothesis. \nResult - Severity of Illness is significantly associated with Stay duration.")
else:
    print("\nFail to Reject the Null Hypothesis. \nResult - No significant association found.")

##### Which statistical test have you done to obtain P-Value?

Chi-Square Test of Independence

##### Why did you choose the specific statistical test?

Both Severity_of_Illness and Stay are categorical variables (Stay is ordinal, but for this test we treat it as categorical buckets), and the Chi-Square Test of Independence is the standard method for testing whether two categorical variables are associated, by comparing observed frequencies against what we'd expect if they were truly independent. With a p-value of essentially 0 (far below the 0.05 threshold) and a Chi-Square statistic of 6616.48, we reject the null hypothesis — this statistically confirms what Chart 9 showed visually: Severity of Illness and Length of Stay are significantly associated, not independent.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H0): Ward Type and Length of Stay (Stay) are independent categorical variables.

Alternate Hypothesis (H1): Ward Type and Length of Stay (Stay) are significantly dependent on each other.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value: Chi-Square Test of Independence
contingency_table = pd.crosstab(df_clean['Ward_Type'], df_clean['Stay'])
chi2, p_val_chi2, dof, expected = chi2_contingency(contingency_table)

print("Chi-Square Test of Independence Results:")
print(f"Chi2 Statistic = {chi2:.4f}, Degrees of Freedom = {dof}, p-value = {p_val_chi2:.4e}")

if p_val_chi2 < 0.05:
    print("\nReject the Null Hypothesis. \nResult - Strong statistical dependence between Ward Type and Stay duration.")
else:
    print("Fail to Reject Null Hypothesis (H0).")


##### Which statistical test have you done to obtain P-Value?

Chi-Square Test of Independence

##### Why did you choose the specific statistical test?

Both `Ward_Type` and `Stay` are categorical variables (Stay is ordinal, but for this test, we treat it as categorical buckets). The Chi-Square Test of Independence is the standard statistical test to determine if there is a significant association between two categorical variables, by comparing the observed frequencies in a contingency table with the frequencies that would be expected if the variables were independent. The very low p-value obtained from the test indicates a strong statistical dependence between Ward Type and Stay duration.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H0): Available extra rooms in hospitals do not differ significantly across hospital region codes .

Alternate Hypothesis (H1): Available extra rooms differ significantly across hospital region codes.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value: One-way ANOVA
rooms_X = df_clean[df_clean['Hospital_region_code'] == 'X']['Available_Extra_Rooms_in_Hospital']
rooms_Y = df_clean[df_clean['Hospital_region_code'] == 'Y']['Available_Extra_Rooms_in_Hospital']
rooms_Z = df_clean[df_clean['Hospital_region_code'] == 'Z']['Available_Extra_Rooms_in_Hospital']

f_stat_reg, p_val_reg = f_oneway(rooms_X, rooms_Y, rooms_Z)
print(f"One-Way ANOVA for Regional Extra Rooms:")
print(f"F-Statistic = {f_stat_reg:.4f}, p-value = {p_val_reg:.4e}")

if p_val_reg < 0.05:
    print("Reject Null Hypothesis (H0).  \nResult - Extra room capacity varies significantly across hospital regions.")
else:
    print("Fail to Reject Null Hypothesis (H0).")


##### Which statistical test have you done to obtain P-Value?

One-Way ANOVA test.

##### Why did you choose the specific statistical test?

The One-Way ANOVA test was chosen because the dependent variable, `Available_Extra_Rooms_in_Hospital`, is a continuous numerical variable, and the independent variable, `Hospital_region_code`, is a categorical variable with three independent groups (X, Y, Z). ANOVA is the appropriate statistical test for comparing the means of a continuous variable across three or more independent categorical groups to determine if there are any statistically significant differences between them.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
print(df_clean.isnull().sum())

#### What all missing value imputation techniques have you used and why did you use those techniques?

Dataset is 100% free of missing values.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
def outlier_handle(data,column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3-Q1
    lower_bound = Q1 - 1.5  IQR
    upper_bound = Q3 + 1.5  IQR
    data[column] = np.clip(data[column], lower_bound, upper_bound)
    return data

df_clean = outlier_handle(df_clean,'Admission_Deposit')
df_clean = outlier_handle(df_clean,'Visitors_with_Patient')

##### What all outlier treatment techniques have you used and why did you use those techniques?

IQR Winsorization/Capping was applied to Admission_Deposit and Visitors_with_Patient. This prevents extreme boundary values from skewing linear coefficients while preserving all patient rows.

### 3. Categorical Encoding & Feature Selection

In [ ]:
# Ordinal Encoding
df_enc = df_clean.copy()

age_mapping = {'0-10': 5, '11-20': 15, '21-30': 25, '31-40': 35, '41-50': 45,
                '51-60': 55, '61-70': 65, '71-80': 75, '81-90': 85, '91-100': 95, 'More than 100 Days': 110}
stay_mapping = {'0-10': 0, '11-20': 1, '21-30': 2, '31-40': 3, '41-50': 4,
                '51-60': 5, '61-70': 6, '71-80': 7, '81-90': 8, '91-100': 9, 'More than 100 Days': 10}
severity_order = {'Minor': 0, 'Moderate': 1, 'Extreme': 2}

df_enc['Age'] = df_clean['Age'].map(age_mapping)
df_enc['Stay_class'] = df_clean['Stay'].map(stay_mapping)
df_enc.drop(['Stay'],axis=1, inplace=True)
df_enc['Severity_of_Illness'] = df_clean['Severity_of_Illness'].map(severity_order)

In [ ]:
# OneHot Encoding
onehot_cols = ['Hospital_type_code','Hospital_region_code','Department', 'Ward_Type', 'Ward_Facility_Code', 'Type_of_Admission']
df_enc = pd.get_dummies(df_enc, columns=onehot_cols, drop_first=True)
df_enc.head()

#### What all categorical encoding techniques have you used & why did you use those techniques?

We used two main categorical encoding techniques:

1.  **Ordinal Encoding:** Applied to `Age`, `Stay`, and `Severity_of_Illness`.
    *   **Why?** These variables possess an inherent, meaningful order (e.g., '0-10' is less than '11-20', 'Minor' severity is less than 'Moderate'). Ordinal encoding assigns a unique integer to each category while preserving this natural order, allowing the model to interpret these relationships correctly without assuming arbitrary distances between categories.

2.  **One-Hot Encoding:** Applied to `Hospital_type_code`, `Hospital_region_code`, `Department`, `Ward_Type`, `Ward_Facility_Code`, and `Type_of_Admission`.
    *   **Why?** These are nominal categorical variables, meaning they have no intrinsic order. One-hot encoding converts each category into a new binary (0 or 1) column. This prevents the model from incorrectly inferring an ordinal relationship between categories and treats each category as an independent feature, which is crucial for algorithms that assume numerical inputs like tree-based models and logistic regression.

### 4. Feature Selection

In [ ]:
corr_matrix = df_enc.corr()

# Reshape into pairwise series and remove self-correlations (diagonal = 1.0)
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1)]

# Drop duplicate pairs (e.g., keeping only one of (A, B) vs (B, A))
unique_pairs = corr_pairs.groupby(frozenset).first()
unique_pairs.index = corr_pairs.index[::2]  # restore clean index

In [ ]:
target_corr = corr_matrix['Stay_class'].drop(['Stay_class'], errors='ignore')
# Least correlated with target (closest to 0)
least_relevant = target_corr[target_corr.abs() < 0.01].sort_values(key=abs)           # default: ascending is True
print("Least correlated with target:")
print(least_relevant)

In [ ]:
df_enc.drop(['Ward_Type_U','Hospital_type_code_e','City_Code_Hospital','Hospital_region_code_Y',
             'Hospital_region_code_Z','Ward_Type_T','City_Code_Patient'],axis=1,inplace=True)

We used a **correlation-based feature selection** method.

**Why?**

1.  **Identify and Remove Irrelevant Features**: We calculated the Pearson correlation coefficient between all features and the target variable (`Stay_class`). Features with very low absolute correlation (in this case, less than 0.01) were considered to have minimal predictive power for the target. Removing these features helps in reducing noise and the complexity of the model.
2.  **Dimensionality Reduction**: By dropping features like `Ward_Type_U`, `Hospital_type_code_e`, `City_Code_Hospital`, `Hospital_region_code_Y`, `Hospital_region_code_Z`, `Ward_Type_T`, and `City_Code_Patient`, we reduce the number of input variables. This can lead to faster training times and can mitigate the risk of overfitting, especially with a large number of one-hot encoded columns. The assumption is that these features do not carry significant information relevant to predicting the length of stay.

##### Which all features you found important and why?

Based on the correlation analysis and domain understanding, the following features were found to be important for predicting patient length of stay:

1.  **Severity_of_Illness**: This is intuitively a highly critical factor. As confirmed by Hypothesis Statement 1 (Chi-Square Test), there is a significant association between severity and `Stay` duration. More severe illnesses often necessitate longer hospitalization, more complex treatments, and closer monitoring.

2.  **Age**: Patient age groups are important. While not directly a cause of longer stays, certain age demographics (e.g., very young or elderly patients, and specifically middle-aged groups as seen in Chart 4) might have different health profiles, recovery times, or co-morbidities that influence their length of stay. The ordinal encoding preserves this inherent order.

3.  **Type_of_Admission**: Different admission types (Emergency, Trauma, Urgent) reflect varying urgency and patient conditions at entry. Chart 7 showed that Trauma admissions, for example, consistently tend to result in longer stays compared to Emergency or Urgent admissions. This indicates distinct clinical pathways and resource needs.

4.  **Visitors_with_Patient**: Chart 8 illustrated a clear positive correlation: an increasing number of average visitors correlates with longer hospital stays. This could be an indicator of social support for patients with more prolonged recovery needs or complex health issues.

5.  **Department**: The medical department (e.g., Gynecology, Anesthesia, Surgery) dictates the type of care, procedures, and conditions being treated. Different departments inherently deal with patient cases that have varying typical lengths of stay, making this a strong predictor.

6.  **Ward_Type** and **Ward_Facility_Code**: These features provide information about the specific care environment. Certain ward types might be dedicated to more critical or long-term care patients (e.g., Ward S and P for extended acute cases, as seen in Chart 11), influencing `Stay` duration. The facility code could also reflect resource availability or specialization.

7.  **Hospital_code** and **Hospital_type_code**: These features capture hospital-specific characteristics that might influence length of stay, such as hospital size, specializations, operational efficiencies, or patient demographics served. Different hospital types (e.g., teaching hospitals vs. general hospitals) may have different average lengths of stay for similar conditions.

8.  **Patient_Admission_Count**: This engineered feature (number of previous admissions for a patient) could indicate patients with chronic conditions or frequent readmissions, which might correlate with longer overall hospital stays or more complex care requirements.

**Why these are important:**
These features were kept because they exhibited a correlation (above an absolute threshold of 0.01) with the target variable (`Stay_class`). This statistical relationship, combined with logical reasoning and visual insights from the EDA, suggests that they carry predictive power. Retaining these features allows the models to capture crucial aspects of the patient's condition, hospital context, and admission circumstances that are influential in determining the length of their hospital stay.

### 5. Data Splitting


In [ ]:
x = df_enc.drop(['Stay_class'],axis=1)
y = df_enc['Stay_class']

x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    random_state=42,
    test_size=0.2)

##### What data splitting ratio have you used and why?

I used a **0.2 test size (80/20 split)** to divide the dataset. This ratio is widely accepted because it gives the model **enough data (80%) to learn patterns effectively** while keeping a **sizable test set (20%) to evaluate performance**. It ensures a good balance between training accuracy and reliable generalization, avoiding overfitting or under-testing.

### 6. Data Scaling

In [ ]:
# Standard Scaling for continuous numerical columns
col1 = x_train.columns
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_train)

col2 = x_test.columns
X_test_scaled = scaler.fit_transform(x_test)

##### Which method have you used to scale you data and why?

You used StandardScaler to scale your data, which is a good choice. It standardizes features by removing the mean and scaling to unit variance. This ensures all numeric variables are on the same scale, preventing features with larger ranges (like deposits) from dominating the model. It also helps algorithms like regression, SVM, and gradient boosting converge faster and perform more reliably.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Yes, the dataset is significantly imbalanced. The training-set class counts range from only 2,190 samples (class 6, 61-70 days) up to 69,888 samples (class 2, 21-30 days) -- an imbalance ratio of about 31.9:1 between the largest and smallest classes. The three middle-length-of-stay buckets (11-20, 21-30, 31-40 days) dominate the data, while longer stays (61-70, 71-80, 81-90, 91-100 days) are rare. A classifier trained naively on this distribution will be biased toward predicting the majority buckets and will struggle to recall the rarer, and often clinically most important, long-stay cases.

In [ ]:
class_counts = y_train.value_counts().sort_index()
print("Class counts in training set:")
print(class_counts)
print(f"\nImbalance ratio (largest class / smallest class): {class_counts.max() / class_counts.min():.2f}")

To address the severe class imbalance, a **class weighting strategy** was employed within the machine learning models.

**Technique:**
For models that support it (like Logistic Regression, Random Forest, and LightGBM), the `class_weight='balanced'` parameter was set. This parameter automatically computes weights inversely proportional to class frequencies in the input data, effectively giving more importance to the minority classes during the training process.

**Why?**
1.  Prevents Bias
2.  Improves Recall and F1-Score
3.  Ensures Generalization

While XGBoost does not have a direct `class_weight` parameter, its objective function and the flexibility of tuning parameters like `scale_pos_weight` (for binary classification, or careful use of weighted metrics for multi-class) serve a similar purpose to handle imbalance. In this project, the evaluation of XGBoost also relied on `f1_weighted` to reflect its performance across all classes fairly.

## ***7. ML Model Implementation***

### ML Model - 1 : Logistic Regression

In [ ]:
# Logistic Regression Model
lg = LogisticRegression()
lg.fit(X_train_scaled, y_train)
y_pred_lg = lg.predict(X_test_scaled)

metrics_lg = pd.DataFrame({
    'Metric': ['Accuracy', 'Weighted Precision', 'Weighted Recall', 'Weighted F1'],
    'Score': [
        accuracy_score(y_test, y_pred_lg),
        precision_score(y_test, y_pred_lg, average='weighted', zero_division=0),
        recall_score(y_test, y_pred_lg, average='weighted'),
        f1_score(y_test, y_pred_lg, average='weighted')
    ]
})
metrics_lg

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart for Logistic Regression

plt.figure(figsize=(8, 4))
sns.barplot(x='Metric', y='Score', data=metrics_lg, palette='Greens_d')
plt.ylim(0, 1)
plt.title('ML Model - 1: Logistic Regression Performance Metrics', fontweight='bold')
for p in plt.gca().patches:
    plt.gca().annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', xytext=(0, 3), textcoords='offset points')
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

param_grid_log_reg = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],  # l2 works with the 'lbfgs' solver we're using; keeps things stable
    'solver': ['lbfgs']
}

# Using cv=3 (not the usual 5) to keep runtime manageable given dataset size
grid_search_lg = GridSearchCV(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1),
    param_grid_log_reg,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

grid_search_lg.fit(x_train, y_train)

print("Best parameters found:", grid_search_lg.best_params_)
print("Best cross-validated F1-score:", grid_search_lg.best_score_)

##### Which hyperparameter optimization technique have you used and why?

GridSearchCV

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

No, there is no noticeable improvement after applying GridSearchCV for hyperparameter tuning to the Logistic Regression model.

### ML Model - 2 : Random Forest

In [ ]:
rf = RandomForestClassifier(class_weight='balanced')
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

metrics_rf = pd.DataFrame({
    'Metric': ['Accuracy', 'Weighted Precision', 'Weighted Recall', 'Weighted F1'],
    'Score': [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf, average='weighted', zero_division=0),
        recall_score(y_test, y_pred_rf, average='weighted'),
        f1_score(y_test, y_pred_rf, average='weighted')
    ]
})
metrics_rf

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart for Random Forest

plt.figure(figsize=(8, 4))
sns.barplot(x='Metric', y='Score', data=metrics_rf, palette='Greens_d')
plt.ylim(0, 1)
plt.title('ML Model - 2: Random Forest Performance Metrics', fontweight='bold')
for p in plt.gca().patches:
    plt.gca().annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', fontsize=10, xytext=(0, 3), textcoords='offset points')
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

param_dist_rf = {
    'n_estimators': [100, 150],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

random_search_rf = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=1),  # single-threaded here
    param_distributions=param_dist_rf,
    n_iter=10,          # reduced from 15 to lighten memory load
    cv=3,
    scoring='f1_weighted',
    n_jobs=4,           # limited parallelism instead of -1 (all cores)
    random_state=42,
    verbose=1
)

random_search_rf.fit(x_train, y_train)

print("Best parameters found:", random_search_rf.best_params_)
print("Best cross-validated F1-score:", random_search_rf.best_score_)

##### Which hyperparameter optimization technique have you used and why?

GridSearchCV was chosen as it provides a structured, reliable way to tune hyperparameters and improve accuracy while maintaining robustness.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

No, there is no noticeable improvement after applying GridSearchCV for hyperparameter tuning to the Random Forest model.

### ML Model - 3 : XGBoost

In [ ]:
# ML Model - 3 Implementation
xgb_model = XGBClassifier(class_weight='balanced')
xgb_model.fit(x_train, y_train)

# Predicting on the test set
y_pred_xgb = xgb_model.predict(x_test)

metrics_xgb = pd.DataFrame({
    'Metric': ['Accuracy', 'Weighted Precision', 'Weighted Recall', 'Weighted F1'],
    'Score': [
        accuracy_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0),
        recall_score(y_test, y_pred_xgb, average='weighted'),
        f1_score(y_test, y_pred_xgb, average='weighted')
    ]
})
metrics_xgb

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
plt.figure(figsize=(8, 4))
sns.barplot(x='Metric', y='Score', data=metrics_xgb, palette='Greens_d')
plt.ylim(0, 1)
plt.title('ML Model - 3:  XGBoost Performance Metrics', fontsize=13, fontweight='bold')
for p in plt.gca().patches:
    plt.gca().annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
param_dist_xgb = {
    'n_estimators': [100, 150, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

random_search_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=42, n_jobs=1, eval_metric='mlogloss'),  # single-threaded inner model
    param_distributions=param_dist_xgb,
    n_iter=10,
    cv=3,
    scoring='f1_weighted',
    n_jobs=4,   # limited parallelism at the search level only
    random_state=42,
    verbose=1
)

# Passing sample_weight through fit_params since XGBoost needs it explicitly
random_search_xgb.fit(x_train, y_train)

print("Best parameters found:", random_search_xgb.best_params_)
print("Best cross-validated F1-score:", random_search_xgb.best_score_)

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

No, there is no noticeable improvement after applying GridSearchCV for hyperparameter tuning to the XGBoost model.

### ML Model - 4 : LightGBM

In [ ]:
# ML Model - 4 Implementation: LightGBM Classifier
lgb_model = LGBMClassifier(class_weight='balanced')
lgb_model.fit(x_train, y_train)
y_pred_lgb = lgb_model.predict(x_test)


metrics_lgb = pd.DataFrame({
    'Metric': ['Accuracy', 'Weighted Precision', 'Weighted Recall', 'Weighted F1'],
    'Score': [
        accuracy_score(y_test, y_pred_lgb),
        precision_score(y_test, y_pred_lgb, average='weighted', zero_division=0),
        recall_score(y_test, y_pred_lgb, average='weighted'),
        f1_score(y_test, y_pred_lgb, average='weighted')
    ]
})

metrics_lgb

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart for LightGBM

plt.figure(figsize=(8, 4))
sns.barplot(x='Metric', y='Score', data=metrics_lgb, palette='Oranges_d')
plt.ylim(0, 1)
plt.title('ML Model - 4: LightGBM Performance Metrics', fontsize=13, fontweight='bold')
for p in plt.gca().patches:
    plt.gca().annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')
plt.show()


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn =KNeighborsClassifier()
knn.fit(X_train_scaled,y_train)
y_pred_knn = knn.predict(X_test_scaled)

metrics_knn = pd.DataFrame({
    'Metric': ['Accuracy', 'Weighted Precision', 'Weighted Recall', 'Weighted F1'],
    'Score': [
        accuracy_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_knn, average='weighted', zero_division=0),
        recall_score(y_test, y_pred_knn, average='weighted'),
        f1_score(y_test, y_pred_knn, average='weighted')
    ]
})

metrics_knn

### Model Comparison Report

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Assume y_test and predictions from each model are available
models = {
    "Logistic Regression": y_pred_lg,
    "Random Forest": y_pred_rf,
    "XGBoost": y_pred_xgb,
    "LightGBM": y_pred_lgb
}

results = []
for name, preds in models.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, average="weighted"),
        "Recall": recall_score(y_test, preds, average="weighted"),
        "F1": f1_score(y_test, preds, average="weighted")
                    })

comparison = pd.DataFrame(results)

comparison = comparison.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)

comparison

#### 1. Which Evaluation metrics did you consider for a positive business impact and why?

| Metric | Purpose | Formula | Business Relevance |
|--------|---------|---------|------------------|
| **Accuracy** | Overall correctness across all classes | (TP + TN) / Total | Shows overall model performance, but less useful with class imbalance |
| **Precision** | Reliability of positive predictions | TP / (TP + FP) | Minimizes false alarms when predicting long stays (avoids unnecessary resource allocation) |
| **Recall** | Ability to find all positive cases | TP / (TP + FN) | Critical for identifying actual long-stay patients; missing these has high cost (resource shortage) |
| **F1-Score** | Harmonic mean of Precision & Recall | 2 × (Precision × Recall) / (Precision + Recall) | Balanced metric for imbalanced data; ensures both false negatives and false positives are minimized |

**Why These Metrics?**  
Given the severe class imbalance in hospital stay prediction, these four metrics together provide a comprehensive view: Accuracy alone would be misleading (model could score 85%+ by predicting majority class). Precision prevents unnecessary resource allocation; Recall catches critical long-stay cases; F1-Score balances both for overall model reliability.

#### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The **XGBoost Classifier** was chosen as the final prediction model for this project.

**Why?**

1.  **Superior F1-Score**: The primary reason for selecting XGBoost was its highest **weighted F1-score** among all tested models (Logistic Regression, Random Forest, and LightGBM). In a dataset with severe class imbalance, the F1-score is a crucial metric as it provides a harmonic mean of precision and recall, ensuring that the model performs well not only on the majority classes but also effectively identifies the minority classes (e.g., long hospital stays) which are often clinically more important.

2.  **Robustness and Performance**: XGBoost is a powerful gradient boosting framework known for its efficiency and accuracy. Its ensemble nature allows it to capture complex non-linear relationships within the data, leading to better generalization and predictive power compared to simpler models like Logistic Regression or even other tree-based models.

3.  **Handling Imbalance**: Although it doesn't have a direct `class_weight` parameter like some other models, its boosting mechanism and the ability to tune parameters effectively allowed it to perform strongly even with the imbalanced target variable. The high F1-score demonstrates its capability to manage the trade-off between false positives and false negatives across all classes.

This makes XGBoost the most reliable and impactful model for accurately predicting patient length of stay, which directly translates to improved hospital resource management and patient care planning.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
pickle.dump(xgb_model, open("House_price_model.pkl", "wb"))

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the trained pipeline from pkl file
model = pickle.load(open("House_price_model.pkl", "rb"))
print(model)

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**


This project successfully built a **multi-class ordinal classification model** to predict hospital length of stay across 11 categories with 318,438 patient records. Despite a severe 31:1 class imbalance, the ensemble methods (XGBoost, LightGBM) significantly outperformed baseline Logistic Regression through weighted evaluation metrics and class-balancing strategies.

**Key findings**: Severity of illness and admission type are strong predictors of stay duration; trauma patients consistently require longer hospitalization. The **weighted F1-score** proved essential for capturing rare long-stay cases clinically important for resource planning.

**Business impact**: Hospitals can now optimize bed allocation, forecast staffing dynamically, and set deposit policies informed by predicted stay duration. A production-ready **data pipeline** (data_pipeline.py) enables seamless processing of new admission records for both model retraining and real-time inference.

**Future work**: Integrate temporal features, deploy as microservice, and monitor model drift in production to maintain predictive accuracy over time.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***

# **KEY CHALLENGES**

1. **Severe Class Imbalance**: The target distribution is heavily right-skewed.  models getting biased toward predicting the majority classes (21-30 days) and will fail to recall rare but clinically important long-stay cases. This requires **balanced evaluation metrics** (weighted F1, recall per class) and **class-weighting strategies** rather than raw accuracy.

2. **Non-linear Feature Relationships**: Preliminary exploratory analysis suggests that severity and admission type show complex interactions with stay duration (e.g., Trauma admissions consistently produce longer stays). Simple linear models may underfit; ensemble methods like Random Forest, XGBoost, and LightGBM will likely outperform Logistic Regression.

3. **Ordinal Nature of Target**: The target is ordered categorical, not nominal. A stay of 41-50 days is "closer" to 31-40 days than to 0-10 days. Standard classification approaches ignore this ordering. **Ordinal encoding** (instead of one-hot) and ordinal-aware evaluation metrics are needed.

4. **Missing & Erroneous Data**:
   - **Bed_Grade**: 113 missing values (0.04%) → Mode imputation
   - **City_Code_Patient**: 4,532 missing values (1.42%) → Mode imputation
   - **Data Entry Errors**: 'Nov-20' typo in Age and Stay columns (likely Excel auto-formatting artifact) → Manual correction required

5. **Feature Redundancy**: Some features show very low correlation with the target (Available_Extra_Rooms_in_Hospital, City_Code_Patient, Ward_Facility_Code) and may introduce noise. **Correlation-based feature selection** will be employed to improve model generalization.

